# Package a UPAR 2027 Track 1 ensemble submission (EVA02-L/14 + SigLIP 2 So400m/14)

Run this notebook from top to bottom. It does **not** train anything: it takes the two single-model submission ZIPs produced by [`upar-track1-eva02-l14.ipynb`](../ViT/upar_track1_eva02_l14/upar-track1-eva02-l14.ipynb) and [`upar-track1-siglip2-so400m14.ipynb`](../CLIP-Style/upar_track1_siglip2_so400m14/upar-track1-siglip2-so400m14.ipynb), puts both fine-tuned checkpoints into one package with a combined `run.py`, and writes `upar_track1_ensemble_eva02_l14_siglip2_so400m14_submission.zip` for Codabench. Neither Internet nor a GPU is needed; packaging and the smoke test take a few minutes on a CPU.

**Inputs.** `upar_track1_eva02_l14_submission.zip` and `upar_track1_siglip2_so400m14_submission.zip`. In Kaggle, attach the outputs of the two training notebooks (or a dataset holding the two ZIPs) with **Add Input**; they are found anywhere under `/kaggle/input`. Outside Kaggle, put the two ZIPs next to this notebook. Kaggle unpacks uploaded ZIPs, so an extracted submission folder (`run.py` and `assets/<model>_best.pth`) is accepted as well.

**Ensemble.** For an image $x$ and its horizontal flip $\tilde{x}$, each model $m$ gives the flip-averaged logits $\bar{z}^{(m)} = \tfrac{1}{2}\big(z^{(m)}(x) + z^{(m)}(\tilde{x})\big)$. The two models are averaged with equal weight in logit space, and the 0.5 threshold of the evaluation is unchanged:

$$p_j(x) = \sigma\!\left(\frac{\bar{z}^{\text{EVA02}}_j + \bar{z}^{\text{SigLIP 2}}_j}{2}\right), \qquad \hat{y}_j = \mathbb{1}\big[p_j(x) > 0.5\big].$$

Each image is decoded once and preprocessed separately for each model: EVA02 uses the CLIP mean and standard deviation, SigLIP 2 uses mean and standard deviation 0.5. Both models were fine-tuned only on the provided training split, so the ensemble uses no additional data. The ZIP is about 1.35 GB (the sum of the two single-model ZIPs), and inference takes about as long as the two single-model runs together.

## 1. Configuration

`MEMBER_TAGS` are the `MODEL_TAG` values of the two training notebooks; they give the names of the input ZIPs and checkpoints. The packaged `run.py` in section 3 is written for exactly these two architectures.

In [1]:
"""Kaggle packaging script for a UPAR 2027 Track 1 ensemble of EVA02-L/14 and SigLIP 2 So400m/14.

Reads the two single-model submission ZIPs and writes one ensemble submission ZIP. Nothing is trained.
This file is also the source for the packaging cells in the accompanying notebook.
"""

import importlib.util
import shutil
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image


OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ".")
SEARCH_ROOTS = (OUTPUT_ROOT, Path("."), Path("/kaggle/input"))  # Where the single-model ZIPs are looked up.
MODEL_TAG = "upar_track1_ensemble_eva02_l14_siglip2_so400m14"
MEMBER_TAGS = ("upar_track1_eva02_l14", "upar_track1_siglip2_so400m14")
SEED = 42

## 2. Locate the single-model submissions

Each member is a folder with the model's own `run.py` and `assets/<tag>_best.pth`. A ZIP is extracted into a temporary folder; the checkpoints are stored in float16, as written by the training notebooks.

In [2]:
def find_member(tag, work):
    """Return a folder that holds the member's run.py and assets/<tag>_best.pth."""
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for archive_path in sorted(root.rglob(f"{tag}_submission.zip")):
            folder = work / tag
            with zipfile.ZipFile(archive_path) as archive:
                assert {"run.py", f"assets/{tag}_best.pth"} <= set(archive.namelist()), archive.namelist()
                archive.extractall(folder)
            print(f"{tag}: {archive_path} ({archive_path.stat().st_size / 1e6:.0f} MB)")
            return folder
        # Kaggle unpacks uploaded ZIPs, so an extracted single-model submission folder is accepted too.
        for checkpoint in sorted(root.rglob(f"{tag}_best.pth")):
            folder = checkpoint.parent.parent
            if (folder / "run.py").is_file() and len(list(checkpoint.parent.glob("*.pth"))) == 1:
                print(f"{tag}: {folder}")
                return folder
    raise FileNotFoundError(
        f"Missing {tag}_submission.zip. Attach the output of the {tag} training notebook in Kaggle "
        f"(Add Input), or put the ZIP next to this notebook. Searched: {[str(root) for root in SEARCH_ROOTS]}"
    )


def load_run(name, folder):
    spec = importlib.util.spec_from_file_location(name, Path(folder) / "run.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def write_sample_images(folder):
    """The challenge images are not downloaded here, so the smoke test uses synthetic crops of varying size."""
    rng = np.random.default_rng(SEED)
    paths = []
    for index, (width, height, mode) in enumerate([(64, 128, "RGB"), (48, 160, "RGB"), (128, 256, "L"), (200, 180, "RGBA")]):
        ys, xs = np.mgrid[0:height, 0:width]
        gradient = np.stack([xs * 255 / width, ys * 255 / height, (xs + ys) * 255 / (width + height)], axis=-1)
        pixels = np.clip(gradient + rng.normal(0, 40, (height, width, 3)), 0, 255).astype(np.uint8)
        path = Path(folder) / f"{index}.png"
        Image.fromarray(pixels).convert(mode).save(path)
        paths.append(path)
    return paths

## 3. Build the inference package

The generated `run.py` exposes the Track 1 API: `load_model()`, `predict_image(sample)` and `predict_batch(samples)`. It defines EVA02 and the SigLIP 2 image encoder in plain PyTorch (timm is not in the evaluation image), with the same code as the two single-model `run.py` files, loads both checkpoints, averages the logits of each image and its horizontal flip per model, averages the two models, and returns 40 probabilities in the order of `sample["attribute_names"]`. `metadata.yaml` is the unchanged file from `examples/task1/sample_code_submission`.

The smoke test runs the packaged `run.py` from the ZIP and checks that its output equals the logit average of the two original single-model `run.py` files, which the training notebooks already checked against timm.

In [3]:
RUN_PY = '''"""UPAR 2027 Track 1 inference entry point (ensemble of EVA02-L/14 and SigLIP 2 So400m/14).

The two fine-tuned models are run on every image and its horizontal flip, and their logits are
averaged with equal weight before the sigmoid. timm is not installed in the evaluation image, so
both networks are defined here in plain PyTorch. Parameter names match timm's Eva and
VisionTransformer, so the fine-tuned state dicts load strictly.
"""
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torchvision import transforms

HERE = Path(__file__).resolve().parent
EVA02_CHECKPOINT = HERE / "assets/upar_track1_eva02_l14_best.pth"
SIGLIP_CHECKPOINT = HERE / "assets/upar_track1_siglip2_so400m14_best.pth"
EVA02_MEAN = (0.48145466, 0.4578275, 0.40821073)  # CLIP normalization used by EVA02.
EVA02_STD = (0.26862954, 0.26130258, 0.27577711)
SIGLIP_MEAN = (0.5, 0.5, 0.5)  # SigLIP normalization: pixels scaled to [-1, 1].
SIGLIP_STD = (0.5, 0.5, 0.5)
BATCH_SIZE = 64
MEMBERS = None  # One (model, transform, attribute_names) tuple per ensemble member.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class PatchEmbed(nn.Module):
    def __init__(self, patch_size, dim):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch_size, patch_size)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)


# ---------------------------------------------------------------------------------------------
# EVA02-L/14
# ---------------------------------------------------------------------------------------------
def rotary_embedding(grid_size, head_dim, ref_grid_size=16, temperature=10000.0):
    """timm RotaryEmbeddingCat (in_pixels=False): sin and cos of 2D patch positions, concatenated."""
    num_bands = head_dim // 4
    bands = 1.0 / temperature ** (torch.arange(num_bands, dtype=torch.float32) / num_bands)
    # EVA02 scales patch coordinates to its 224 / 14 = 16 x 16 pretraining grid.
    t = torch.arange(grid_size, dtype=torch.float32) / grid_size * ref_grid_size
    grid = torch.stack(torch.meshgrid(t, t, indexing="ij"), dim=-1).unsqueeze(-1)
    positions = (grid * bands).reshape(grid_size * grid_size, -1)
    return torch.cat([positions.sin().repeat_interleave(2, -1), positions.cos().repeat_interleave(2, -1)], -1)


def apply_rotary(x, embedding):
    sin, cos = embedding.chunk(2, -1)
    rotated = torch.stack([-x[..., 1::2], x[..., ::2]], -1).reshape(x.shape)  # (x0, x1) -> (-x1, x0)
    return x * cos + rotated * sin


class RotaryAttention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x, rope):
        b, n, c = x.shape
        q, k, v = (
            projection(x).reshape(b, n, self.num_heads, -1).transpose(1, 2)
            for projection in (self.q_proj, self.k_proj, self.v_proj)
        )
        # Rotary position embeddings rotate the patch tokens only, not the class token.
        q = torch.cat([q[:, :, :1], apply_rotary(q[:, :, 1:], rope)], dim=2).type_as(v)
        k = torch.cat([k[:, :, :1], apply_rotary(k[:, :, 1:], rope)], dim=2).type_as(v)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(b, n, c))


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1_g = nn.Linear(dim, hidden_dim)
        self.fc1_x = nn.Linear(dim, hidden_dim)
        self.act = nn.SiLU()
        self.norm = nn.LayerNorm(hidden_dim, eps=1e-6)
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.norm(self.act(self.fc1_g(x)) * self.fc1_x(x)))


class EVA02Block(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = RotaryAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = SwiGLU(dim, hidden_dim)

    def forward(self, x, rope):
        x = x + self.attn(self.norm1(x), rope)
        return x + self.mlp(self.norm2(x))


class EVA02(nn.Module):
    """timm Eva for EVA02: RoPE, SwiGLU with sub-LayerNorm, average pooling of patch tokens."""

    def __init__(self, patch_size, num_patches, dim, depth, num_heads, hidden_dim, num_classes):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, dim))
        self.blocks = nn.ModuleList([EVA02Block(dim, num_heads, hidden_dim) for _ in range(depth)])
        self.fc_norm = nn.LayerNorm(dim, eps=1e-6)
        self.head = nn.Linear(dim, num_classes)
        self.register_buffer("rope", rotary_embedding(math.isqrt(num_patches), dim // num_heads), persistent=False)

    def forward(self, x):
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(len(x), -1, -1), x], dim=1) + self.pos_embed
        for block in self.blocks:
            x = block(x, self.rope)
        return self.head(self.fc_norm(x[:, 1:].mean(dim=1)))


# ---------------------------------------------------------------------------------------------
# SigLIP 2 So400m/14
# ---------------------------------------------------------------------------------------------
class Mlp(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()  # timm uses the exact GELU.
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class Attention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        b, n, c = x.shape
        q, k, v = self.qkv(x).reshape(b, n, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(b, n, c))


class SigLIPBlock(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, hidden_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class AttentionPool(nn.Module):
    """timm AttentionPoolLatent: a learned query attends to all tokens, then an MLP with a residual."""

    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()
        self.num_heads = num_heads
        self.latent = nn.Parameter(torch.zeros(1, 1, dim))
        self.q = nn.Linear(dim, dim)
        self.kv = nn.Linear(dim, dim * 2)
        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, hidden_dim)

    def forward(self, x):
        b, n, c = x.shape
        q = self.q(self.latent.expand(b, -1, -1)).reshape(b, 1, self.num_heads, -1).transpose(1, 2)
        k, v = self.kv(x).reshape(b, n, 2, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        x = self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(b, 1, c))
        x = x + self.mlp(self.norm(x))
        return x[:, 0]


class SigLIPViT(nn.Module):
    """timm VisionTransformer for SigLIP: no class token, final norm, attention pooling (MAP)."""

    def __init__(self, patch_size, num_patches, dim, depth, num_heads, hidden_dim, num_classes):
        super().__init__()
        self.patch_embed = PatchEmbed(patch_size, dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, dim))
        self.blocks = nn.ModuleList([SigLIPBlock(dim, num_heads, hidden_dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.attn_pool = AttentionPool(dim, num_heads, hidden_dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x) + self.pos_embed
        for block in self.blocks:
            x = block(x)
        return self.head(self.attn_pool(self.norm(x)))


# ---------------------------------------------------------------------------------------------
# Loading and inference
# ---------------------------------------------------------------------------------------------
def load_member(checkpoint_path, mean, std, has_class_token):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    size = int(checkpoint["image_size"])
    transform = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    state = checkpoint["state_dict"]
    arguments = dict(
        patch_size=state["patch_embed.proj.weight"].shape[-1],
        dim=state["pos_embed"].shape[2],
        depth=len({key.split(".")[1] for key in state if key.startswith("blocks.")}),
        num_heads=int(checkpoint["num_heads"]),
        num_classes=state["head.weight"].shape[0],
    )
    if has_class_token:  # EVA02: pos_embed includes the class token.
        model = EVA02(
            num_patches=state["pos_embed"].shape[1] - 1,
            hidden_dim=state["blocks.0.mlp.fc1_g.weight"].shape[0],
            **arguments,
        )
    else:
        model = SigLIPViT(
            num_patches=state["pos_embed"].shape[1],
            hidden_dim=state["blocks.0.mlp.fc1.weight"].shape[0],
            **arguments,
        )
    model.load_state_dict(state)
    return model.float().to(DEVICE).eval(), transform, checkpoint["attribute_names"]


def load_model():
    global MEMBERS
    MEMBERS = [
        load_member(EVA02_CHECKPOINT, EVA02_MEAN, EVA02_STD, has_class_token=True),
        load_member(SIGLIP_CHECKPOINT, SIGLIP_MEAN, SIGLIP_STD, has_class_token=False),
    ]


def predict_batch(samples):
    if MEMBERS is None:
        load_model()
    images = []
    for sample in samples:
        with Image.open(sample["image_path"]) as image:
            images.append(image.convert("RGB"))
    names = samples[0]["attribute_names"]
    logits = 0.0
    for model, transform, attribute_names in MEMBERS:
        batch = torch.stack([transform(image) for image in images]).to(DEVICE)
        with torch.inference_mode(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            member_logits = (model(batch).float() + model(torch.flip(batch, dims=[3])).float()) / 2
        # Each checkpoint stores its own attribute order, so align it before averaging.
        logits = logits + member_logits[:, [attribute_names.index(name) for name in names]]
    probabilities = (logits / len(MEMBERS)).sigmoid().clamp(0.0, 1.0).cpu().numpy()
    probabilities = np.nan_to_num(probabilities, nan=0.0)
    rows = []
    for sample, row in zip(samples, probabilities):
        order = [names.index(name) for name in sample["attribute_names"]]
        rows.append(row[order].astype(float).tolist())
    return rows


def predict_image(sample):
    return predict_batch([sample])[0]
'''

METADATA_YAML = '''# Code submissions are executed by the challenges ingestion program, which
# imports run.py and calls predict_image() / predict_batch(). Nothing in this
# file is used for inference, it exists only so the platform never treats the
# archive as a result submission. Leave it as-is.
command: python3 -c "print('Submission package ready. The ingestion program imports run.py')"
'''


def package_submission(members):
    folder = OUTPUT_ROOT / f"{MODEL_TAG}_submission"
    assets = folder / "assets"
    assets.mkdir(parents=True, exist_ok=True)
    (folder / "run.py").write_text(RUN_PY)
    (folder / "metadata.yaml").write_text(METADATA_YAML)
    for tag, member in members.items():
        shutil.copy2(member / "assets" / f"{tag}_best.pth", assets / f"{tag}_best.pth")
    archive_path = OUTPUT_ROOT / f"{MODEL_TAG}_submission.zip"
    expected = {"run.py", "metadata.yaml"} | {f"assets/{tag}_best.pth" for tag in members}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in sorted(expected):
            archive.write(folder / name, name)
    with zipfile.ZipFile(archive_path) as archive:
        print("ZIP contents:", archive.namelist())
        assert set(archive.namelist()) == expected
        with tempfile.TemporaryDirectory() as temp_dir:
            archive.extractall(temp_dir)
            module = load_run("upar_submission_check", temp_dir)
            module.load_model()
            names = list(module.MEMBERS[0][2])
            samples = [
                {"index": i, "image": str(path), "image_path": str(path), "attribute_names": names}
                for i, path in enumerate(write_sample_images(temp_dir))
            ]
            batch = np.asarray(module.predict_batch(samples))
            single = np.asarray(module.predict_image(samples[0]))
            assert batch.shape == (len(samples), 40) and single.shape == (40,)
            assert np.isfinite(batch).all() and ((batch >= 0) & (batch <= 1)).all()
            # A different attribute order in the sample must give the same values, reordered.
            shuffled = [names[i] for i in np.random.default_rng(SEED).permutation(40)]
            reordered = np.asarray(module.predict_image({**samples[0], "attribute_names": shuffled}))
            assert np.abs(reordered - single[[names.index(name) for name in shuffled]]).max() < 1e-3
            print("Packaged inference smoke test passed")
            module.MEMBERS.clear()
            # The ensemble must equal the logit average of the two single-model run.py files.
            member_probabilities = []
            for tag, member in members.items():
                member_module = load_run(f"{tag}_check", member)
                member_module.load_model()
                member_probabilities.append(np.asarray(member_module.predict_batch(samples), dtype=np.float64))
                member_module.MODEL = None
            # Saturated probabilities (exactly 0 or 1 in float32) cannot be turned back into logits.
            valid = np.all([(p > 1e-6) & (p < 1 - 1e-6) for p in member_probabilities], axis=0)
            mean_logit = np.mean([np.log(p[valid]) - np.log1p(-p[valid]) for p in member_probabilities], axis=0)
            difference = np.abs(batch[valid] - 1 / (1 + np.exp(-mean_logit))).max()
            assert valid.mean() > 0.5 and difference < 1e-3, f"ensemble and single-model outputs differ by {difference}"
            print(f"run.py matches the logit average of the single-model submissions (max probability difference {difference:.1e})")
            for tag, probabilities in zip(members, member_probabilities):
                print(f"Attributes predicted present per image, {tag}: {(probabilities > 0.5).sum(axis=1).tolist()}")
            print(f"Attributes predicted present per image, ensemble: {(batch > 0.5).sum(axis=1).tolist()}")
    print(f"ZIP size: {archive_path.stat().st_size / 1e6:.0f} MB")
    print("Upload this ZIP to Codabench:", archive_path)

In [4]:
def main():
    torch.manual_seed(SEED)
    with tempfile.TemporaryDirectory() as work:
        members = {tag: find_member(tag, Path(work)) for tag in MEMBER_TAGS}
        package_submission(members)

## 4. Create the ZIP

After this cell finishes, find the submission ZIP under `/kaggle/working` (or next to this notebook outside Kaggle). The leaderboard shows the **latest** submission, not the best one, so resubmit the best single-model ZIP if the ensemble scores lower.

In [5]:
main()

upar_track1_eva02_l14: upar_track1_eva02_l14_submission.zip (561 MB)


upar_track1_siglip2_so400m14: upar_track1_siglip2_so400m14_submission.zip (791 MB)


ZIP contents: ['assets/upar_track1_eva02_l14_best.pth', 'assets/upar_track1_siglip2_so400m14_best.pth', 'metadata.yaml', 'run.py']


Packaged inference smoke test passed


run.py matches the logit average of the single-model submissions (max probability difference 1.3e-07)
Attributes predicted present per image, upar_track1_eva02_l14: [5, 5, 5, 6]
Attributes predicted present per image, upar_track1_siglip2_so400m14: [8, 7, 8, 12]
Attributes predicted present per image, ensemble: [8, 7, 6, 7]
ZIP size: 1352 MB
Upload this ZIP to Codabench: upar_track1_ensemble_eva02_l14_siglip2_so400m14_submission.zip
